# Scrapy & Anti-Bot — Security Lab (Red Team vs Blue Team)

This is a **security education** notebook: we dissect *how a real-world scraper defeats the
anti-bot protection* of an actual e-commerce site, **and then** map out how to detect and prevent
each technique. The goal is not "to steal data," but to help **engineers and blue teams understand
the attack** so they can build the right defenses.

Case study: `https://www.blibli.com/c3/olahraga-sepeda/AK-1000051`

> Every "red" (offensive) technique below is always paired with a
> **"Prevention (Blue Team)"** section. That pairing is the whole point of the lesson.

## 0. Lab Rules & Ethics (must read)

- **For education and defense only.** Do not use this to harm anyone.
- We touch only **public data** (no login) and send the **fewest requests possible** (just a
  handful), with delays — this is **not** a mass-harvest.
- Defeating anti-bot protection can **violate Terms of Service** and, at certain scales or with
  certain consequences, **break the law** (e.g. Indonesia's ITE Law, the US CFAA). In the real
  world: ask for permission, use an official API, or buy licensed data.
- This notebook is deliberately "loud" about defenses so that you can **close these gaps** in
  your own systems.

## Glossary — key terms

- **User-Agent (UA):** the HTTP header string a client sends to identify its software (e.g. "Chrome on macOS").
- **Cookie:** a small piece of data the server sets in your browser and that is sent back on later requests.
- **Session:** a sequence of related requests tied together by shared cookies/state, like one browsing visit.
- **robots.txt:** a file where a site declares which paths automated crawlers are allowed or disallowed to fetch.
- **SPA (Single-Page App):** a site (often React) whose content is loaded by JavaScript via APIs after the initial page load, so it isn't in the raw HTML.
- **WAF (Web Application Firewall):** a security layer that inspects and filters incoming traffic to block attacks and abuse.
- **Bot management:** WAF features that score and challenge traffic to tell automated bots apart from real users.
- **TLS fingerprint / JA3 / JA4:** the unique signature of how a client negotiates the HTTPS (TLS) handshake; JA3 and JA4 are hashes summarizing that signature.
- **Impersonation (e.g. `curl_cffi`):** making a non-browser client's low-level signature (such as its TLS handshake) look exactly like a real browser's.
- **Warm-up request:** an initial request to a page made only to collect the cookies a real browser would have, before hitting the actual target.
- **Managed / JS challenge:** a defense that forces the client to execute JavaScript (or solve a check) before a cookie/session is treated as valid.
- **Headless browser:** a real browser engine (e.g. Chromium via Playwright) running with no visible window, used to execute JS and render pages programmatically.
- **Rate limiting:** capping how many requests a client (IP/cookie/account) may make in a time window to curb abuse.
- **IP reputation:** scoring an IP by its history and network type (e.g. datacenter vs residential) to decide how much to trust it.
- **Honeytoken:** a hidden, fake item (link, field, or value) that only an automated scraper would grab, used to detect and flag bots.

In [1]:
# --- Shared setup for the whole notebook ---
BROWSER_UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
              "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
PAGE = "https://www.blibli.com/c3/olahraga-sepeda/AK-1000051"
CAT  = "AK-1000051"

def api_url(page=1, n=8):
    return ("https://www.blibli.com/backend/search/products"
            f"?categoryId={CAT}&page={page}&itemPerPage={n}&channelId=web")

# Headers that mimic Chrome (used across several levels)
BROWSER_HEADERS = {
    "User-Agent": BROWSER_UA,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,id;q=0.8",
    "sec-ch-ua": '"Chromium";v="124", "Google Chrome";v="124", "Not-A.Brand";v="99"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"macOS"',
    "Upgrade-Insecure-Requests": "1",
}
print("Setup ready. Target:", PAGE)

Setup ready. Target: https://www.blibli.com/c3/olahraga-sepeda/AK-1000051


## 1. Recon — "Are we allowed?" and "Is it possible?"

Before attacking or scraping anything, professionals **always do recon first**: check the
`robots.txt`, the page type (static vs SPA — a Single-Page App whose content is loaded by
JavaScript after the initial page load), and the kind of protection in place.

In [2]:
import requests
import urllib.robotparser as urobot
from protego import Protego  # the robots.txt parser used by Scrapy

# Fetch robots.txt with a browser UA (Python's default UA often gets a 403 -> parser misreads it)
robots = requests.get("https://www.blibli.com/robots.txt",
                      headers={"User-Agent": BROWSER_UA}, timeout=20).text
rp = urobot.RobotFileParser(); rp.parse(robots.splitlines())
pr = Protego.parse(robots)

paths = {
    "Category page": "/c3/olahraga-sepeda/AK-1000051",
    "Product API (data source)": "/backend/search/products",
    "Detail page": "/p/product-name/ps--ABC-12345",
}
print(f"{'PATH':28} | urllib | Protego(Scrapy)")
print("-" * 60)
for label, path in paths.items():
    a = "ALLOWED" if rp.can_fetch("*", path) else "DISALLOWED"
    b = "ALLOWED" if pr.can_fetch(path, "*") else "DISALLOWED"
    print(f"{label:28} | {a:6} | {b}")

PATH                         | urllib | Protego(Scrapy)
------------------------------------------------------------
Category page                | ALLOWED | ALLOWED
Product API (data source)    | ALLOWED | DISALLOWED
Detail page                  | ALLOWED | DISALLOWED


Finding: **the `/backend/search/*` API is `DISALLOWED`** (Protego/Scrapy is correct; `urllib` gets
it wrong because it's weak at handling the `*` wildcard). Yet that endpoint is exactly the source
of the product list. In other words: according to `robots.txt`, the data path is **not permitted**.
(Note this down for the ethics discussion later.)

In [3]:
# Page type & protection type: inspect the response headers & cookies
r = requests.get(PAGE, headers=BROWSER_HEADERS, timeout=20)
print("Category page      : HTTP", r.status_code, "|", len(r.text), "bytes")
print("Server header      :", r.headers.get("Server"))
print("cf-ray (Cloudflare):", r.headers.get("cf-ray"))
print("Notable Set-Cookie :", [c for c in r.cookies.keys()])
print("Products in HTML?  : 'olahraga-sepeda' appears",
      r.text.count("olahraga-sepeda"), "x (data is loaded via API, not SSR)")

# Try the API directly, with no tricks at all
a = requests.get(api_url(), headers={"User-Agent": BROWSER_UA, "Accept": "application/json"}, timeout=20)
print("\nAPI with no tricks : HTTP", a.status_code,
      "->", "BLOCKED" if a.status_code != 200 else "passed")

Category page      : HTTP 200 | 186207 bytes
Server header      : cloudflare
cf-ray (Cloudflare): a05ab9a5fd654ab5-CGK
Notable Set-Cookie : ['__cf_bm', '_cfuvid']
Products in HTML?  : 'olahraga-sepeda' appears 0 x (data is loaded via API, not SSR)



API with no tricks : HTTP 403 -> BLOCKED


**Target profile:**

- **SPA** (React) — the product list is **not** in the HTML; it's loaded via the API after the
  page renders.
- Sitting behind **Cloudflare Bot Management** (a Web Application Firewall / WAF feature that
  scores and filters automated traffic — see `cf-ray` + the `__cf_bm` cookie).
- The data API returns **403** for ordinary clients, and is **`DISALLOWED` by `robots.txt`**.

This is why people often say "Blibli can't be scraped." It actually can — by stacking up bypass
techniques. Let's dissect them one by one (red), then how to shut each one down (blue).

## 2. The Escalation Ladder — Red vs Blue

Each level adds one more trick. We watch **when the API starts to let us through**, and **what
the defense is** at each step.

> ⚠️ **A WAF is *stateful*.** Cloudflare scores you based on IP + session + timing. Because the
> recon cells above already "warmed up" our IP, the middle levels (L1/L2) **sometimes already
> pass** even though a "cold" IP would normally get a 403. So **read the actual status that gets
> printed**, not what you memorized — that is precisely the important lesson. What is **always**
> consistent: L0 (fails) and L3 (the full recipe, succeeds).

### Level 0 — plain `requests` (Python UA)

The most naive approach: GET the API directly. This is what a beginner bot does. The
**User-Agent (UA)** — the header that identifies the client software — is the giveaway here.

In [4]:
import requests
try:
    r = requests.get(api_url(), timeout=20)  # default UA: "python-requests/x"
    print("HTTP", r.status_code, "| content-type:", r.headers.get("content-type"))
except Exception as e:
    print("ERR", e)
print("=> Blocked. The 'python-requests' UA is recognized instantly.")

HTTP 403 | content-type: text/html; charset=UTF-8
=> Blocked. The 'python-requests' UA is recognized instantly.


**Prevention (Blue Team):** block or immediately flag **non-browser User-Agents** and empty UAs;
this is the cheapest detection there is. Don't rely on it alone (it's trivial to fake).

### Level 1 — Full browser headers

The scraper adds a Chrome `User-Agent` plus `Accept`, `Accept-Language`, `sec-ch-ua`, and so on.

In [5]:
r = requests.get(api_url(), headers={**BROWSER_HEADERS, "Accept": "application/json",
                                     "Referer": PAGE}, timeout=20)
ok = r.status_code == 200 and "json" in r.headers.get("content-type", "")
print("HTTP", r.status_code, "| content-type:", r.headers.get("content-type"))
print("=> Passed: browser headers make the request look more legitimate." if ok
      else "=> Still blocked: headers alone aren't enough; the WAF checks more than headers.")
print("   (Remember: a WAF is stateful — the result depends on how 'warm' the IP/session is right now.)")

HTTP 200 | content-type: application/json
=> Passed: browser headers make the request look more legitimate.
   (Remember: a WAF is stateful — the result depends on how 'warm' the IP/session is right now.)


**Prevention (Blue Team):** *header anomaly detection* — check for consistency (e.g. `sec-ch-ua`
must match the UA; real browsers send headers in a characteristic order). Still easy to fake, so
you need the next layer.

### Level 2 — TLS fingerprint (JA3/JA4)

The key secret: **`requests`/Scrapy have a distinctly Python TLS fingerprint** (the cipher order
and extensions used during the HTTPS handshake) that is **different from Chrome's**. A TLS
fingerprint (the unique signature of how a client negotiates HTTPS) is summarized as a hash called
**JA3 or JA4**. WAFs like Cloudflare/Akamai match this JA3/JA4 — so even if the UA says "Chrome,"
the TLS says "Python" → **the lie is exposed**.

The red-team weapon: **`curl_cffi`**, which can **impersonate Chrome's TLS** (`impersonate="chrome"`)
— impersonation means making the client's low-level signature look like a real browser's.

In [6]:
from curl_cffi import requests as creq
# Impersonate Chrome's TLS, but WITHOUT a cookie warm-up yet
r = creq.get(api_url(), impersonate="chrome",
             headers={"Referer": PAGE, "Accept": "application/json"}, timeout=20)
ok = r.status_code == 200 and "json" in r.headers.get("content-type", "")
print("HTTP", r.status_code, "| content-type:", r.headers.get("content-type"))
print("=> Chrome TLS closes the JA3 gap — harder to tell apart from a real browser." if ok
      else "=> Chrome TLS alone isn't enough on this endpoint: it needs a cookie warm-up (Level 3).")

HTTP 200 | content-type: application/json
=> Chrome TLS closes the JA3 gap — harder to tell apart from a real browser.


**Prevention (Blue Team):** **fingerprint the TLS handshake (JA3/JA4)** and reject or challenge on
a mismatch (UA="Chrome" but TLS≠Chrome). This is what defeats many scrapers — and why attackers
move on to `curl_cffi`/`utls`/a real browser.

### Level 3 — The full recipe: cookie warm-up + TLS impersonation (reliably GETS THROUGH)

The most reliable recipe: **visit the page first** like a real browser to **obtain Cloudflare's
bot-management cookie** (`__cf_bm`) — this initial "warm-up request" exists only to collect the
cookies a browser would have — then use that cookie (within the same *session*, with Chrome TLS)
to call the API.

In [7]:
from curl_cffi import requests as creq
import json

s = creq.Session(impersonate="chrome")           # Chrome TLS for every request
warm = s.get(PAGE, headers=BROWSER_HEADERS, timeout=20)   # warm-up -> obtain cookies
print("Warm-up page    :", warm.status_code, "| cookies:", list(s.cookies.keys()))

resp = s.get(api_url(n=8), headers={"Referer": PAGE, "Accept": "application/json"}, timeout=20)
print("API after warmup:", resp.status_code, "| content-type:", resp.headers.get("content-type"))

if resp.status_code == 200 and "json" in resp.headers.get("content-type", ""):
    products = resp.json()["data"]["products"]
    print(f"\nGOT THROUGH — {len(products)} real products:")
    for p in products[:5]:
        print(f"  - {p['name'][:55]:55} | {p['price']['priceDisplay']}")

Warm-up page    : 200 | cookies: ['__cf_bm', '_cfuvid']


API after warmup: 200 | content-type: application/json

GOT THROUGH — 8 real products:
  - United Detroit 1121 Sepeda Gunung / MTB 27.5 12 Speed F | Rp4.200.000
  - Strider 14x Footrest - Black                            | Rp210.000
  - Strider 14x Clamp - Black                               | Rp50.000
  - United Detroit 1101 Sepeda Gunung / MTB 27.5 10 Speed F | Rp3.000.000
  - SEPEDA GUNUNG ELEMENT COYOTE MTB SPY 26 INCH 7 SPEED    | Rp1.695.000


**Chrome TLS + cookie warm-up combined = through.** This is the technique commonly used in the
real world against cookie-based WAFs.

**Prevention (Blue Team):**
- **Managed Challenge / JS Challenge**: force JavaScript execution before a cookie becomes valid
  (so a cookie from a plain GET is not honored). `curl_cffi` doesn't run JS, so this stops it.
- **Bind the cookie to the fingerprint + IP + time**; short TTL; detect cookies reused across
  different IPs.
- **Rate-limit per cookie/IP** and run behavioral analytics (see Level 5).

### Level 4 — A real browser (Playwright/Selenium) *(concept)*

If the WAF requires **JS execution** (a managed challenge) or human-like behavior, the attacker
escalates to a **headless browser** (a real browser engine running with no visible window) that
executes the sensor JS and renders the SPA. This is often combined with **stealth** (hiding
`navigator.webdriver`, etc.).

```python
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(user_agent=BROWSER_UA)
    page.goto(PAGE, wait_until="networkidle")     # run the sensor JS -> valid cookie
    # Capture the API response the page fires, or read the already-rendered DOM:
    data = page.evaluate("() => window.fetch(API_URL).then(r => r.json())")
    browser.close()
```

**Prevention (Blue Team):**
- **Detect headless/automation**: `navigator.webdriver`, missing properties, render timing.
- **Canvas/WebGL/AudioContext fingerprinting** + **behavioral biometrics** (mouse movement, rhythm).
- **CAPTCHA / managed challenge** when the risk score is high.

### Level 5 — At scale: proxy & UA rotation *(concept)*

For high volume, the attacker spreads requests across **many IPs (residential proxies)** and
**rotates the UA/fingerprint** to avoid per-IP rate limiting.

```python
# Scrapy: proxy middleware + UA rotation (illustration)
DOWNLOADER_MIDDLEWARES = {
    "scrapy_proxies.RandomProxy": 610,
    "myproject.middlewares.RandomUserAgent": 400,
}
ROTATING_PROXY_LIST = ["http://ip1:port", "http://ip2:port", ...]
```

**Prevention (Blue Team):**
- **Rate limiting & velocity checks** per IP/account/cookie; **IP reputation** (block datacenter
  ASNs / known proxies).
- **Volume/pattern anomalies** (time of day, non-human page sequences) → challenge or block.
- **Honeypot/honeytoken**: a hidden link or price field that only a bot would grab → flag it.

## 3. Putting it together in **Scrapy** (the "hacky" version that actually runs)

Now we package the **Level 3** technique into a production-style Scrapy spider using
[`scrapy-impersonate`](https://github.com/jxlil/scrapy-impersonate) (a download handler built on
`curl_cffi` → Chrome TLS). The spider:

1. **Warms up** by hitting the category page (the `__cf_bm` cookie is stored automatically in
   Scrapy's cookie jar).
2. **Calls the API** `/backend/search` page by page (pagination) and parses the JSON.
3. Normalizes the price with **`price-parser`** and emits it as an **`Item`**.

> A "hacky" note: we set **`ROBOTSTXT_OBEY = False`** — which technically goes against Blibli's
> `robots.txt`. **This is deliberate**, as a talking point: this is the decision that moves the
> activity from "polite scraping" into a "gray/illegal area." In your own systems, this is exactly
> what you should anticipate.

In [8]:
spider_code = r"""
import scrapy, json
from price_parser import Price

UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
CAT = "AK-1000051"
PAGE = "https://www.blibli.com/c3/olahraga-sepeda/" + CAT


class ProductItem(scrapy.Item):
    name       = scrapy.Field()
    price_text = scrapy.Field()
    price      = scrapy.Field()   # float from price-parser
    brand      = scrapy.Field()
    rating     = scrapy.Field()
    sold_count = scrapy.Field()
    merchant   = scrapy.Field()
    url        = scrapy.Field()


class BlibliSpider(scrapy.Spider):
    name = "blibli"
    max_page = 2  # keep the demo small: polite & lightweight

    custom_settings = {
        "ROBOTSTXT_OBEY": False,                 # <- the 'hacky' decision (see the note)
        "USER_AGENT": UA,
        "DEFAULT_REQUEST_HEADERS": {
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9,id;q=0.8",
            "sec-ch-ua": '\"Chromium\";v=\"124\", \"Google Chrome\";v=\"124\", \"Not-A.Brand\";v=\"99\"',
            "sec-ch-ua-mobile": "?0",
            "sec-ch-ua-platform": '\"macOS\"',
        },
        # Chrome TLS via scrapy-impersonate (the key to passing JA3)
        "DOWNLOAD_HANDLERS": {
            "https": "scrapy_impersonate.ImpersonateDownloadHandler",
            "http": "scrapy_impersonate.ImpersonateDownloadHandler",
        },
        "TWISTED_REACTOR": "twisted.internet.asyncioreactor.AsyncioSelectorReactor",
        "CONCURRENT_REQUESTS": 1,     # polite
        "DOWNLOAD_DELAY": 1.0,        # human-like delay
        "AUTOTHROTTLE_ENABLED": True,
        "LOG_LEVEL": "INFO",
        "FEED_EXPORT_ENCODING": "utf-8",
    }

    def api(self, page):
        return (f"https://www.blibli.com/backend/search/products"
                f"?categoryId={CAT}&page={page}&itemPerPage=12&channelId=web")

    async def start(self):
        # Level 3: warm up the page first to obtain the __cf_bm cookie
        yield scrapy.Request(PAGE, meta={"impersonate": "chrome"}, callback=self.after_warmup)

    def after_warmup(self, response):
        self.logger.info(f"WARMUP status={response.status}")
        yield scrapy.Request(self.api(1), meta={"impersonate": "chrome", "page": 1},
                             headers={"Referer": PAGE, "Accept": "application/json"},
                             callback=self.parse_api)

    def parse_api(self, response):
        page = response.meta["page"]
        data = json.loads(response.text).get("data", {})
        products = data.get("products", [])
        self.logger.info(f"APISTATUS page={page} status={response.status} products={len(products)}")

        for p in products:
            price = Price.fromstring(p["price"].get("priceDisplay", ""))
            rev = p.get("review") or {}
            it = ProductItem()
            it["name"]       = p.get("name")
            it["price_text"] = p["price"].get("priceDisplay")
            it["price"]      = price.amount_float
            it["brand"]      = p.get("brand")
            it["rating"]     = rev.get("absoluteRating")
            it["sold_count"] = p.get("soldCountTotal")
            it["merchant"]   = p.get("merchantName")
            it["url"]        = "https://www.blibli.com" + (p.get("url") or "")
            yield it

        # limited pagination
        if products and page < self.max_page:
            nxt = page + 1
            yield scrapy.Request(self.api(nxt), meta={"impersonate": "chrome", "page": nxt},
                                 headers={"Referer": PAGE, "Accept": "application/json"},
                                 callback=self.parse_api)
"""

with open("/tmp/blibli_spider.py", "w") as f:
    f.write(spider_code)
print("Spider written to /tmp/blibli_spider.py")

Spider written to /tmp/blibli_spider.py


### Run the spider (via subprocess)

In [9]:
import subprocess, sys, os, time

OUT = "/tmp/blibli_hasil.json"
if os.path.exists(OUT):
    os.remove(OUT)

t0 = time.time()
res = subprocess.run(
    [sys.executable, "-m", "scrapy", "runspider", "/tmp/blibli_spider.py", "-O", OUT],
    capture_output=True, text=True, timeout=180,
)
duration = time.time() - t0

for line in res.stderr.splitlines():
    if any(k in line for k in ["WARMUP", "APISTATUS", "item_scraped_count",
                                "robotstxt", "finish_reason"]):
        print(line.strip()[-140:])
print(f"\nReturncode: {res.returncode} | Duration: {duration:.1f}s")

2026-06-03 07:54:15 [blibli] INFO: WARMUP status=200
2026-06-03 07:54:22 [blibli] INFO: APISTATUS page=1 status=200 products=12
2026-06-03 07:54:25 [blibli] INFO: APISTATUS page=2 status=200 products=12
'finish_reason': 'finished',
'item_scraped_count': 24,

Returncode: 0 | Duration: 11.4s


### Results → `pandas` (ready for cleaning/DB)

In [10]:
import pandas as pd, json

with open("/tmp/blibli_hasil.json") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print("Number of products:", len(df))
print("\nData types:")
print(df.dtypes)
if len(df):
    print(f"\nPrice: cheapest Rp{df['price'].min():,.0f} | "
          f"most expensive Rp{df['price'].max():,.0f} | average Rp{df['price'].mean():,.0f}")
df[["name", "price", "brand", "rating", "sold_count", "merchant"]].head(10)

Number of products: 24

Data types:
name              str
price_text        str
price         float64
brand             str
rating        float64
sold_count     object
merchant          str
url               str
dtype: object

Price: cheapest Rp20,300 | most expensive Rp4,320,000 | average Rp1,928,885


,name,price,brand,rating,sold_count,merchant
0,United Detroit 1121 Sepeda Gunung / MTB 27.5 1...,4200000.0,United Bike,0.0,None,Serba Sepeda
1,Strider 14x Footrest - Black,210000.0,Strider,0.0,None,Strider Bikes
2,Strider 14x Clamp - Black,50000.0,Strider,0.0,None,Strider Bikes
3,United Detroit 1101 Sepeda Gunung / MTB 27.5 1...,3000000.0,United Bike,0.0,None,Serba Sepeda
4,SEPEDA GUNUNG ELEMENT COYOTE MTB SPY 26 INCH 7...,1695000.0,Element,0.0,None,tokohappybike Flagship Store
5,UNO Seatpost Sepeda SP358 31.6 400mm Bahan All...,379620.0,UNO,0.0,None,tokohappybike Flagship Store
6,Element Sepeda Hybrid Jasper Ukuran 700C 21 Sp...,2350000.0,ELEMENT BIKE,0.0,None,Toko Dunia Sepeda
7,Element Montreal URB Ukuran 700C Shimano Cues ...,4320000.0,Element,0.0,None,Serba Sepeda
8,SEPEDA HYBRID JASPER ELEMENT HYDRAULIC 8+ 700C...,2750000.0,ELEMENT BIKE,0.0,None,Toko Dunia Sepeda
9,Sepeda Listrik Uwinfly D66A Terbaru 2025 Anti ...,4150000.0,Uwinfly,4.7,None,U-WINFLY INDONESIA Flagship Store


## 4. Blue Team — Layered Defense (summary)

The core lesson: **there is no silver bullet.** A good defense = many layers, each one raising the
attacker's cost.

| Attacker technique (red) | How to detect/prevent it (blue) |
|---|---|
| Non-browser / empty UA (L0) | Block/flag odd UAs; but don't rely on this alone |
| Fake browser headers (L1) | *Header anomaly detection* (`sec-ch-ua` consistency vs UA, header order) |
| TLS impersonation / fake JA3 (L2) | **JA3/JA4 fingerprinting**; reject when TLS≠the claimed UA |
| WAF cookie warm-up (L3) | **Managed/JS challenge** (require JS execution); bind cookies to fingerprint+IP; short TTL |
| Headless browser + stealth (L4) | Detect `navigator.webdriver`, canvas/WebGL fingerprinting, **behavioral biometrics**, CAPTCHA |
| Proxy & UA rotation, at scale (L5) | **Rate limiting & velocity**, **IP reputation** (block datacenter ASNs), volume anomalies |
| All of the above | **Honeytokens**, monitoring & alerting, authentication + request signing for sensitive APIs |

**What Blibli already does well:** SPA + Cloudflare bot-management + a `robots.txt` that disallows
the API + TLS fingerprinting. **What could be strengthened:** a mandatory-JS managed challenge on
the data endpoint (which would break Level 3 in this notebook), plus strict per-cookie rate
limiting.

## Conclusion

- **"Can be scraped" ≠ "may be scraped".** We got through technically, but the data path is
  `DISALLOWED` by `robots.txt` — in the real world, use an **official API / permission / licensed
  data**.
- The educational value: you now understand the **attack chain (L0→L5)** and the **defense at each
  layer**, so you can **secure** your own APIs/sites.
- The techniques & libraries here (`curl_cffi`, `scrapy-impersonate`, Playwright) are **dual-use** —
  use them ethically and legally.